# M63h — standalone reproducibility check
**M63H-STANDALONE-2026-10-02-r3.** Select NVIDIA L4. Run sections1–4 in order; no other notebook needed.
Two fresh processes × three GT-only updates. No KD, full epoch, validation sweep or promoted checkpoint. Existing Drive results remain unchanged. This is not an accuracy-improvement run.
After a runtime reset, rerun1–4. Existing completed M63h reports are reused. An interrupted diagnostic replays from A2. All model commands run in fresh subprocesses.

**r2 runtime repair:** private virtual environment, no Colab shared site-packages; pinned Numba0.61.2/llvmlite0.44.0 uses Numba's bundled CUDA backend with the CUDA12.8 toolkit. Resolved packages are recorded and reused. Historical Numba identity is unknown; this checks repeatability in a repaired runtime, not exact historical equivalence. Results go to diagnostics_m63h_isolated, preserving failed attempts.

r3 bootstrap repair: handles an existing partial private environment and installs pip through the host pip --python option. It does not call ensurepip, delete the environment, or install model dependencies into shared Colab Python.


## 1. Restore environment and source
Mount Drive, pin Python packages, verify frozen source, select/install CUDA12.8 toolkit, and rebuild the extension. The driver is not replaced. No earlier experiment is run. If KITTI lives elsewhere, edit DATASET_CANDIDATES below.


In [ ]:
# M63H-STANDALONE-2026-10-02-r3 — visible notebook revision.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys, zipfile

PYTHON = Path('/content/m63h_cuda128_venv/bin/python')
RUNTIME_RECEIPT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m63_r0_a2_depth_pilot/m63h_isolated_runtime.json')
for key in ('PYTHONPATH','PYTHONHOME','LD_PRELOAD','NUMBA_ENABLE_CUDASIM'):
    os.environ.pop(key,None)
os.environ['PYTHONNOUSERSITE']='1'
os.environ['NUMBA_CUDA_USE_NVIDIA_BINDING']='0'
print('M63H-STANDALONE-2026-10-02-r3')
DRIVE = Path('/content/drive/MyDrive')
MOBILE_REPO = Path('/content/mobile_adas3d')
REPO = Path('/content/MonoDETR_M62')
COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
OUTPUT = DRIVE / 'mobile_adas3d_outputs/students/monodetr_m62_r0_a2_diagnostic'
M62_OUTPUT = OUTPUT
OUTPUT = DRIVE / 'mobile_adas3d_outputs/students/monodetr_m63_r0_a2_depth_pilot'
MANIFEST = OUTPUT / 'm63_manifest.json'
DATASET = Path('/content/kitti_m62')
SPLITS = DRIVE / 'mobile_adas3d_splits/kitti_chen'
R0_SELECTION = DRIVE / 'mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json'
A2_SELECTION = DRIVE / 'mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json'
DATASET_CANDIDATES = [DATASET, Path('/content/kitti_m61'), Path('/content/kitti'),
                      Path('/content/monodetr_kitti'), DRIVE / 'datasets/kitti']

def run(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT / 'colab_logs' / (name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=80)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    with log_path.open('a', buffering=1) as log:
        log.write('\nSTART ' + datetime.now(timezone.utc).isoformat() + '\n')
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
        rc = p.wait()
    if rc:
        raise RuntimeError(f'Exit {rc}; log={log_path}\n' + '\n'.join(tail))

def pilot(script, *args):
    run([str(PYTHON), '-u', MOBILE_REPO / 'scripts' / script,
         '--manifest', MANIFEST, *args], script + '_isolated', MOBILE_REPO)

run(['nvidia-smi'], 'gpu')
M62 = json.loads((M62_OUTPUT / 'm62_manifest.json').read_text())
if M62['repo'] != str(REPO) or M62['dataset_root'] != str(DATASET):
    raise RuntimeError('Restore the original M62 paths; do not edit the frozen manifest.')

PREFLIGHT_READY = False
mobile_url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE_REPO.exists():
    run(['git', 'clone', '--branch', 'main', mobile_url, MOBILE_REPO], 'clone_mobile')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Use main, not {branch!r}')
run(['git', 'pull', '--ff-only'], 'update_mobile', MOBILE_REPO)
run([sys.executable, MOBILE_REPO / 'scripts/setup_m63_isolated_runtime.py',
     '--manifest',M62_OUTPUT/'m62_manifest.json','--venv',PYTHON.parent.parent,
     '--receipt',RUNTIME_RECEIPT],'isolated_dependencies',MOBILE_REPO)
# cv2.__version__ is not necessarily the distribution version (wheel suffix).
# Validate the actual imported versions before rebuilding the native extension.
probe = ("import sys; sys.path.insert(0, sys.argv[1]); "
         "from m61_common import environment; from diagnose_m62_r0_a2 import require_identity; "
         "import json; require_identity(json.load(open(sys.argv[2]))['environment'], environment())")
run([str(PYTHON), '-c', probe, MOBILE_REPO / 'scripts', M62_OUTPUT / 'm62_manifest.json'],
    'verify_m62_environment', MOBILE_REPO)
url = '/'.join(['https:', '', 'github.com', 'ZrrSkywalker', 'MonoDETR.git'])
if not REPO.exists():
    run(['git', 'clone', url, REPO], 'clone_monodetr')
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if head != COMMIT:
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
        raise RuntimeError('Wrong upstream commit with local changes; use a new dedicated directory.')
    run(['git', 'fetch', 'origin', COMMIT], 'fetch_monodetr', REPO)
    run(['git', 'checkout', '--detach', COMMIT], 'pin_monodetr', REPO)
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py',
              'patch_monodetr_mobilenetv4.py'):
    run([str(PYTHON), MOBILE_REPO / 'scripts' / patch, '--monodetr-repo', REPO], patch, MOBILE_REPO)
ops = REPO / 'lib/models/monodetr/ops'
patch_arch = ("import sys; from pathlib import Path; sys.path.insert(0, sys.argv[1]); "
              "from setup_m61_sources import patch_active_architecture; "
              "patch_active_architecture(Path(sys.argv[2]))")
run([str(PYTHON), '-c', patch_arch, MOBILE_REPO / 'scripts', ops / 'setup.py'], 'cuda_arch', MOBILE_REPO)
source_probe = ("import sys,json; from pathlib import Path; sys.path.insert(0,sys.argv[1]); "
                "from m61_common import source_hash; m=json.load(open(sys.argv[2])); "
                "assert (Path(m['repo'])/'lib').is_dir(), 'MonoDETR source directory missing'; "
                "assert source_hash(m['repo'])==m['source_sha256'], 'Restored source differs from frozen M62'")
run([str(PYTHON), '-c', source_probe, MOBILE_REPO / 'scripts',
     M62_OUTPUT / 'm62_manifest.json'], 'verify_restored_source', MOBILE_REPO)
# Select the compiler separately from the PyTorch CUDA runtime.
cuda = Path('/usr/local/cuda-' + M62['environment']['cuda'])
if not (cuda/'bin/nvcc').is_file():
    run(['apt-get','update','-qq'], 'cuda_package_index')
    run(['apt-get','install','-y','cuda-toolkit-' + M62['environment']['cuda'].replace('.','-')],
        'cuda_toolkit_install')
if not (cuda/'bin/nvcc').is_file():
    raise RuntimeError('Matching toolkit unavailable; send cuda_toolkit_install.log. Do not bypass.')
os.environ['CUDA_HOME'] = str(cuda)
os.environ['CUDA_PATH'] = str(cuda)
os.environ['LD_LIBRARY_PATH'] = str(cuda/'lib64') + ':' + str(cuda/'nvvm/lib64')
os.environ['PATH'] = str(cuda/'bin') + ':' + os.environ['PATH']
version = subprocess.check_output([str(cuda/'bin/nvcc'),'--version'],text=True)
print(version)
if 'release ' + M62['environment']['cuda'] + ',' not in version:
    raise RuntimeError('Selected CUDA compiler version differs')
os.environ['MAX_JOBS'] = '2'
run([str(PYTHON), 'setup.py', 'build_ext', '--inplace', '--force'], 'cuda_build', ops)


## 2. Restore data and check all prerequisites
Check all image/label/calibration files and both Chen splits before creating links. Then verify frozen manifests, source, environment, checkpoints, reviewed M63g report and approved targets. Missing dataset files must be supplied; this notebook does not download KITTI.


In [ ]:
# Reconstruct ephemeral dataset paths before any provenance load.
command = [str(PYTHON), MOBILE_REPO/'scripts/restore_m63_data.py',
           '--dataset',DATASET,'--splits',SPLITS]
for candidate in DATASET_CANDIDATES:
    command += ['--candidate',candidate]
run(command,'restore_data',MOBILE_REPO)
probe = ("import sys; sys.path.insert(0,sys.argv[1]); "
         "from m63_common import load_manifest; from train_m63_student import validated_audit; "
         "from diagnose_m63_reproducibility import REVIEWED; from m61_common import sha256; "
         "from pathlib import Path; m=load_manifest(sys.argv[2]); "
         "assert sha256(Path(m['output_dir'])/'diagnostics_m63g/m63g_diagnostic.json')==REVIEWED; "
         "validated_audit(m); print('All frozen provenance and approved-target checks passed')")
run([str(PYTHON),'-c',probe,MOBILE_REPO/'scripts',MANIFEST],'complete_preflight',MOBILE_REPO)
run([str(PYTHON),MOBILE_REPO/'scripts/smoke_m63_isolated_runtime.py',
     '--manifest',MANIFEST,'--receipt',RUNTIME_RECEIPT,
     '--output',OUTPUT/'m63h_isolated_runtime_smoke.json'],'runtime_cuda_smoke',MOBILE_REPO)
PREFLIGHT_READY = True


## 3. Run the bounded M63h diagnostic
Exactly six optimizer updates total, starting from A2 in each fresh process. Temporary tensor snapshots are local and removed after comparison. No deployable checkpoint is saved.


In [ ]:
if not globals().get('PREFLIGHT_READY',False):
    raise RuntimeError('Run sections1 and2 successfully first.')
run([str(PYTHON),'-m','unittest','discover','-s','tests','-p','test_m63_reproducibility.py','-v'],
    'm63h_tests',MOBILE_REPO)
pilot('diagnose_m63_reproducibility.py','--runtime-receipt',RUNTIME_RECEIPT)


## 4. Download and stop
Return m63h_results.zip for review. A completed diagnostic is not permission for longer training.


In [ ]:
DIAG=OUTPUT/'diagnostics_m63h_isolated'
bundle=OUTPUT/'m63h_results.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('identity.json','trace_a.json','trace_b.json','m63h_reproducibility.json'):
        path=DIAG/name
        if not path.is_file(): raise FileNotFoundError(path)
        z.write(path,name)
    z.write(RUNTIME_RECEIPT,'m63h_isolated_runtime.json')
    z.write(OUTPUT/'m63h_isolated_runtime_smoke.json','m63h_isolated_runtime_smoke.json')
print('Return:',bundle,'STOP for review.')
from google.colab import files
files.download(str(bundle))
